# 06_mllm / mllm_inference.ipynb

**Purpose:** Generate free-form natural-language explanations for a stratified sample of 500 GSAF predictions using **Gemini 2.5 Flash-Lite** (free, 1000 RPD).

**Sample design — 500 pairs total:**
- ~150 from each test set (NewsCLIPpings, MMFakeBench, VERITE) where local images exist.
- Within each dataset: balanced across the 4 confusion-matrix buckets (TP / FP / FN / TN).
- Preference for high-confidence predictions (clearer signal for the LLM).

**Free-tier budget:**
- Gemini 2.5 Flash-Lite: 15 RPM, **1000 RPD**.
- 500 calls = ~50 minutes wall clock at full RPM. Easily fits in 1 day; leaves quota for the next notebook.

**Inputs:**
- `cache/fusion/fusion_predictions.csv`
- `data/processed/ooc_pairs/manifests/all.csv`
- `cache/mllm/.mllm_setup_ok` (must exist — produced by `gemini_setup.ipynb`)
- `GEMINI_API_KEY` in `.env`

**Outputs:**
- `cache/mllm/explanations_freeform.csv` — final table
- `cache/mllm/explanations_freeform.jsonl` — append-only raw log
- `paper/tables/mllm_explanations_sample.csv` — top 20 picks for paper appendix

The notebook is restartable — already-cached responses are skipped on re-run, so a Ctrl-C and resume costs zero quota.


In [1]:
# Imports & path setup
from __future__ import annotations
import json, sys, time, warnings, re
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

NB_DIR = Path.cwd()
ROOT = NB_DIR
for parent in [NB_DIR] + list(NB_DIR.parents):
    if (parent / "src" / "utils" / "paths.py").exists():
        ROOT = parent; break
sys.path.insert(0, str(ROOT))

from src.utils.paths import (
    ALL_MANIFEST, FUSION_CACHE, PROCESSED_IMAGES, VERITE_IMAGES,
    PAPER, TABLES, FIGURES,
)

CACHE_MLLM = ROOT / "cache" / "mllm"
CACHE_MLLM.mkdir(parents=True, exist_ok=True)

sentinel = CACHE_MLLM / ".mllm_setup_ok"
if not sentinel.exists():
    raise RuntimeError(f"Run 06_mllm/gemini_setup.ipynb first — missing sentinel at {sentinel}")
print("Setup sentinel:", json.loads(sentinel.read_text()))


Setup sentinel: {'model_name': 'models/gemini-2.5-flash-lite', 'fallback_model': 'models/gemini-2.5-flash', 'provider': 'gemini', 'rpm_limit': 14, 'rpd_limit': 950, 'ts': 1777359699.4961698}


## 1. Load fusion predictions + manifest, compute buckets

In [2]:
thresholds_path = FUSION_CACHE / "fusion_thresholds.json"
fusion_path     = FUSION_CACHE / "fusion_predictions.csv"

# fusion_predictions.csv now contains 5 model variants (default + 4 ablations).
# We only generate explanations for the default GSAF model.
preds_all = pd.read_csv(fusion_path)
preds = preds_all[preds_all["model"] == "GSAF_default"].copy()
print(f"Filtered to GSAF_default: {len(preds):,} rows (of {len(preds_all):,})")

manifest = pd.read_csv(ALL_MANIFEST)
thresholds = json.loads(thresholds_path.read_text()) if thresholds_path.exists() else {}

def find_col(df, cands):
    for c in cands:
        if c in df.columns: return c
    return None

prob_col = find_col(preds, ["score", "fusion_prob", "y_prob"])
y_col    = find_col(preds, ["label", "y_true", "target"])
pid_col  = find_col(preds, ["pair_id", "id"])
es_col   = find_col(preds, ["eval_split"])
ga, gb, gc = (find_col(preds, [c]) for c in ["gate_a", "gate_b", "gate_c"])

def thresh_for(es): return float(thresholds.get(es, thresholds.get("default", 0.5)))
preds["_threshold"] = preds[es_col].map(thresh_for)
preds["_y_pred"] = (preds[prob_col].astype(float) >= preds["_threshold"]).astype(int)
preds["_bucket"] = np.where(
    (preds[y_col] == 1) & (preds["_y_pred"] == 1), "TP",
    np.where((preds[y_col] == 0) & (preds["_y_pred"] == 1), "FP",
    np.where((preds[y_col] == 1) & (preds["_y_pred"] == 0), "FN", "TN")))

ds_col = find_col(preds, ["dataset"])
sp_col = find_col(preds, ["split"])
test_only = preds[preds[es_col].str.contains("test", na=False)]
print("\nBucket counts (test splits):")
print(test_only.groupby([ds_col, "_bucket"]).size())


Filtered to GSAF_default: 25,789 rows (of 128,945)

Bucket counts (test splits):
dataset        _bucket
mmfakebench    FN         1177
               FP          995
               TN         2005
               TP         5823
newsclippings  FN         1335
               FP          940
               TN         2692
               TP         2297
verite         FN           60
               FP          314
               TN           24
               TP          603
dtype: int64


## 2. Resolve image paths, merge captions

In [3]:
merged = preds.merge(
    manifest[["pair_id", "caption", "image_path"]].drop_duplicates("pair_id"),
    on="pair_id", how="left"
)

def resolve_image(p):
    if pd.isna(p) or not p: return None
    pp = Path(str(p))
    if pp.is_absolute() and pp.exists(): return pp
    for base in [PROCESSED_IMAGES, VERITE_IMAGES, ROOT / "data" / "raw" / "mmfakebench", ROOT]:
        cand = base / pp.name
        if cand.exists(): return cand
        cand2 = base / pp
        if cand2.exists(): return cand2
    return None

merged["_image_path"] = merged["image_path"].map(resolve_image)
print("Rows with local image:", merged["_image_path"].notna().sum(), "/", len(merged))
print("By dataset:")
print(merged.groupby(ds_col)["_image_path"].apply(lambda s: s.notna().sum()))


Rows with local image: 11121 / 25789
By dataset:
dataset
mmfakebench      10500
newsclippings        0
verite             621
Name: _image_path, dtype: int64


## 3. Stratified sample — 500 pairs

In [4]:
TOTAL_BUDGET = 500
BUCKETS = ["TP", "FP", "FN", "TN"]

pool = merged[
    merged[es_col].str.contains("test", na=False) &
    merged["_image_path"].notna() &
    merged["caption"].notna()
].copy()
print("Pool size:", len(pool), "rows by dataset:", pool[ds_col].value_counts().to_dict())

datasets_present = pool[ds_col].unique()
PER_DATASET = TOTAL_BUDGET // len(datasets_present)

selected = []
rng = np.random.default_rng(42)

for ds_name in datasets_present:
    ds_pool = pool[pool[ds_col] == ds_name]
    per_bucket = max(1, PER_DATASET // len(BUCKETS))
    for bk in BUCKETS:
        sub = ds_pool[ds_pool["_bucket"] == bk].copy()
        if sub.empty: continue
        # Confidence-weighted score
        if bk in ("TP", "FP"):
            sub["_score"] = sub[prob_col].astype(float)
        else:
            sub["_score"] = 1.0 - sub[prob_col].astype(float)
        sub = sub.sort_values("_score", ascending=False).head(per_bucket * 3)
        idx = rng.choice(sub.index, size=min(per_bucket, len(sub)), replace=False)
        selected.append(sub.loc[idx])

sample_df = pd.concat(selected, ignore_index=True)
print(f"\nSelected {len(sample_df)} pairs:")
print(sample_df.groupby([ds_col, "_bucket"]).size())

keep = [pid_col, ds_col, sp_col, y_col, "_y_pred", "_bucket", prob_col, ga, gb, gc,
        "caption", "_image_path"]
keep = [c for c in keep if c is not None]
sample_df = sample_df[keep].reset_index(drop=True)
sample_df.head()


Pool size: 10621 rows by dataset: {'mmfakebench': 10000, 'verite': 621}

Selected 456 pairs:
dataset      _bucket
mmfakebench  FN         62
             FP         62
             TN         62
             TP         62
verite       FN         60
             FP         62
             TN         24
             TP         62
dtype: int64


,pair_id,dataset,split,label,_y_pred,_bucket,score,gate_a,gate_b,gate_c,caption,_image_path
0,mmfakebench_test_05608,mmfakebench,test,1,1,TP,0.981889,0.671511,0.059578,0.268911,Discovery: NASA Unveils Revolutionary Antigrav...,E:\eve-rag\data\processed\ooc_pairs\images\mmf...
1,mmfakebench_test_05771,mmfakebench,test,1,1,TP,0.984208,0.693773,0.043262,0.262965,"Phoenix, Arizona Emerges as Thriving Canadian ...",E:\eve-rag\data\processed\ooc_pairs\images\mmf...
2,mmfakebench_test_05825,mmfakebench,test,1,1,TP,0.984884,0.656905,0.060494,0.282601,Kenny Chesney: Canadian Country Music Star Mak...,E:\eve-rag\data\processed\ooc_pairs\images\mmf...
3,mmfakebench_test_06852,mmfakebench,test,1,1,TP,0.982666,0.661651,0.068068,0.270280,Robert Downey Jr left and Gwyneth Paltrow star...,E:\eve-rag\data\processed\ooc_pairs\images\mmf...
4,mmfakebench_test_04820,mmfakebench,test,1,1,TP,0.984237,0.692095,0.043676,0.264229,Kim Kardashian 'Hurt' By Awkward Exchange With...,E:\eve-rag\data\processed\ooc_pairs\images\mmf...


## 4. Free-form prompt template

In [5]:
FREEFORM_PROMPT = '''You are a fact-checking analyst. You are given an image and a caption that was published with it.

The caption is:
"""{caption}"""

Your task: assess whether the image plausibly matches the caption.

Write a short response with three labelled lines:

IMAGE: One sentence describing what is visible in the image (objects, setting, people, mood).
CAPTION_CHECK: One sentence stating whether the image plausibly supports the caption, or whether something seems off (mismatched location, wrong people, time-of-day clue, anachronism, etc.).
VERDICT: Exactly one word — either MATCH or MISMATCH.

Be specific. Do not say "I cannot tell" unless the image is genuinely unreadable.'''

print(FREEFORM_PROMPT.format(caption="<EXAMPLE>")[:500])


You are a fact-checking analyst. You are given an image and a caption that was published with it.

The caption is:
"""<EXAMPLE>"""

Your task: assess whether the image plausibly matches the caption.

Write a short response with three labelled lines:

IMAGE: One sentence describing what is visible in the image (objects, setting, people, mood).
CAPTION_CHECK: One sentence stating whether the image plausibly supports the caption, or whether something seems off (mismatched location, wrong people, ti


## 5. Generate explanations

In [ ]:
from src.utils.mllm_client import MLLMClient

client = MLLMClient()
print(f"Model: {client.model_name}  |  daily remaining: {client.daily_remaining()}")
print(f"This run will need up to {len(sample_df)} new calls (cached calls are free).")

OUT_CSV   = CACHE_MLLM / "explanations_freeform.csv"
OUT_JSONL = CACHE_MLLM / "explanations_freeform.jsonl"

done_ids = set()
if OUT_CSV.exists():
    prev = pd.read_csv(OUT_CSV)
    done_ids = set(prev[pid_col].astype(str).tolist())
    print(f"Already on disk: {len(done_ids)}")

records = []
errors = []

for _, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc="Gemini"):
    pid = str(row[pid_col])
    if pid in done_ids: continue

    cap = str(row["caption"])
    img = row["_image_path"]
    prompt = FREEFORM_PROMPT.format(caption=cap[:600])

    try:
        text = client.generate(
            prompt=prompt, image_path=img,
            use_cache=True, temperature=0.2, max_output_tokens=300,
        )
        rec = {
            "pair_id":     pid,
            "dataset":     row[ds_col],
            "split":       row[sp_col],
            "y_true":      int(row[y_col]),
            "y_pred":      int(row["_y_pred"]),
            "bucket":      row["_bucket"],
            "fusion_prob": float(row[prob_col]),
            "gate_a":      float(row[ga]) if ga in row.index and pd.notna(row[ga]) else None,
            "gate_b":      float(row[gb]) if gb in row.index and pd.notna(row[gb]) else None,
            "gate_c":      float(row[gc]) if gc in row.index and pd.notna(row[gc]) else None,
            "caption":     cap,
            "explanation": text,
        }
        records.append(rec)
        with open(OUT_JSONL, "a", encoding="utf-8") as f:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    except RuntimeError as e:
        msg = str(e)
        if "Daily quota" in msg:
            print(f"\n[stop] {msg}")
            break
        errors.append({"pair_id": pid, "error": msg})
        if records:
            pd.DataFrame(records).to_csv(OUT_CSV, index=False)
    except Exception as e:
        errors.append({"pair_id": pid, "error": str(e)})

# Merge with previous run, dedupe
if OUT_CSV.exists():
    prev = pd.read_csv(OUT_CSV)
    new_df = pd.DataFrame(records)
    full = pd.concat([prev, new_df], ignore_index=True).drop_duplicates(pid_col, keep="last")
else:
    full = pd.DataFrame(records)

full.to_csv(OUT_CSV, index=False)
print(f"\nDone this run. Total on disk: {len(full)}")
print(f"Errors this run: {len(errors)}")
if errors:
    print(pd.DataFrame(errors).head())
print(f"Daily quota remaining: {client.daily_remaining()}")


Model: models/gemini-2.5-flash-lite  |  daily remaining: 927
This run will need up to 456 new calls (cached calls are free).


Gemini:   0%|          | 0/456 [00:00<?, ?it/s]

## 6. Parse VERDICT and compare with GSAF

In [ ]:
def parse_verdict(text):
    if not isinstance(text, str): return None
    m = re.search(r"VERDICT\s*:\s*(MATCH|MISMATCH)", text, re.IGNORECASE)
    if m: return m.group(1).upper()
    for line in reversed(text.strip().splitlines()):
        ln = line.strip().upper()
        if "MISMATCH" in ln: return "MISMATCH"
        if "MATCH" in ln: return "MATCH"
    return None

full = pd.read_csv(OUT_CSV)
full["mllm_verdict"] = full["explanation"].map(parse_verdict)
full["mllm_pred_ooc"] = full["mllm_verdict"].map({"MISMATCH": 1, "MATCH": 0})

print("Verdict counts:")
print(full["mllm_verdict"].value_counts(dropna=False))

mask = full["mllm_pred_ooc"].notna()
if mask.any():
    agree = (full.loc[mask, "mllm_pred_ooc"] == full.loc[mask, "y_pred"]).mean()
    mllm_acc = (full.loc[mask, "mllm_pred_ooc"] == full.loc[mask, "y_true"]).mean()
    gsaf_acc = (full.loc[mask, "y_pred"]       == full.loc[mask, "y_true"]).mean()
    print(f"\nOn pairs MLLM gave a verdict for (n={mask.sum()}):")
    print(f"  MLLM ↔ GSAF agreement   : {agree:.3f}")
    print(f"  MLLM zero-shot accuracy : {mllm_acc:.3f}")
    print(f"  GSAF accuracy           : {gsaf_acc:.3f}")

full.to_csv(OUT_CSV, index=False)


## 7. Pick best 20 examples for paper appendix

In [ ]:
def quality(row):
    if pd.isna(row.get("mllm_verdict")): return -1
    score = 0
    text = str(row.get("explanation", ""))
    if 100 < len(text) < 800: score += 2
    if "IMAGE:" in text and "CAPTION_CHECK:" in text: score += 3
    if row["bucket"] in ("FP", "FN", "TP"): score += 1
    score += abs(float(row["fusion_prob"]) - 0.5) * 2
    return score

full["_quality"] = full.apply(quality, axis=1)
top = (full[full["_quality"] > 0]
       .sort_values("_quality", ascending=False)
       .groupby("dataset", group_keys=False).head(7)
       .head(20))

paper_cols = ["pair_id", "dataset", "bucket", "y_true", "y_pred", "mllm_verdict",
              "fusion_prob", "caption", "explanation"]
top[paper_cols].to_csv(TABLES / "mllm_explanations_sample.csv", index=False)
print("Saved:", TABLES / "mllm_explanations_sample.csv")
top[paper_cols].head(10)


---
**Done.** Next: `chain_of_thought.ipynb` runs the same 500 pairs through a structured CoT prompt.